In [ ]:
import math
import shutil
from functools import reduce
from pathlib import Path
from random import random
from time import sleep

import pyarrow as pa
import pyarrow.parquet as pq

import numpy as np
import polars as pl
from jinja2.ext import do
from tqdm import tqdm

from python.process_data import files

In [ ]:
DATA_DIR = Path("/mnt/Fast2T/data/ai-training-set/")
DATASET_DIR = Path(DATA_DIR, "dataset/")

Find embedding
We embedd a text using the usage of stem words, so to find a embedding we combine the top words of:
- web data (to have some relation to our data set)
- ai data (to find patterns like —)
- wikipedia data (to find human patterns)

In [9]:
TOP_K = 1024

ai = pl.read_parquet(DATA_DIR / "total" / "ai.parquet")
website = pl.read_parquet(DATA_DIR / "total" / "web.parquet")
wiki = pl.read_parquet(DATA_DIR / "total" / "human.parquet")
grok = pl.read_parquet(DATA_DIR / "total" / "grok.parquet")
cc = pl.read_parquet(DATA_DIR / "total" / "cc.parquet")

common = (
    ai.select("term", pl.col("freq").alias("len_ai"))
    .join(wiki.select("term", pl.col("freq").alias("len_wiki")), on="term")
    .join(website.select("term", pl.col("freq").alias("len_website")), on="term")
    .join(grok.select("term", pl.col("freq").alias("len_grok")), on="term")
    .join(cc.select("term", pl.col("freq").alias("len_cc")), on="term")
    .unique(subset="term")
)

embedding = (
    common
    .with_columns(pl.mean_horizontal("len_ai", "len_wiki", "len_website", "len_grok", "len_cc").alias("len_avg"))
    .top_k(k=TOP_K, by="len_avg")
    .sort("term")
    .select("term")
    .rename({"term": "token"})
    .with_row_index("id")
)

embedding.write_parquet(DATASET_DIR / "embedding.parquet")

embedding_size = int(embedding.count()["id"][0])
embedding

id,token
u32,str
0,"""about"""
1,"""above"""
2,"""access"""
3,"""accessory"""
4,"""account"""
…,…
1019,"""york"""
1020,"""you"""
1021,"""young"""


Create a dataset based on the embedding

In [2]:
TARGET_DIR = Path(DATA_DIR / "sampled")
TARGET_PER_SOURCE = 4_000_000

for type in ["ai", "grok", "cc", "human", "web"]:
    files = list(Path(DATA_DIR / type).iterdir())
    random.shuffle(files)

    print(files)

    # src_dir = DATA_DIR / "raw-cleaned" / type
    #
    # files = list(src_dir.iterdir())
    # data_point_size = sum(pq.read_metadata(file).num_rows for file in files)
    # keep_per_file = TARGET_PER_SOURCE * 1 / data_point_size
    # print(f"[{type}] {data_point_size} rows, keep_per_file={keep_per_file:.4f}, target={TARGET_PER_SOURCE}")
    #
    # writer = None
    #
    # for file in tqdm(files, desc=type):
    #     df = pl.read_parquet(file)
    #
    #     if keep_per_file < 1:
    #         df = df.sample(fraction=keep_per_file, shuffle=True)
    #
    #     table = df.to_arrow()
    #     del df
    #
    #     if writer is None:
    #         writer = pq.ParquetWriter(
    #             SAMPLED_DIR / f"{type}.parquet",
    #             table.schema,
    #             compression="zstd"
    #         )
    #
    #     writer.write_table(table)
    #     del table
    # writer.close()

NameError: name 'Path' is not defined

In [3]:
# Split chunks into ready size
SAMPLED_DIR = Path(DATA_DIR / "sampled")
SHARDS = 4

for type in ["ai", "grok", "cc", "human", "web"]:
    src = SAMPLED_DIR / f"{type}.parquet"
    n_rows = pq.read_metadata(src).num_rows
    shard_size = math.ceil(n_rows / SHARDS)

    df = pl.scan_parquet(src)

    for i in range(SHARDS):
        (
            df.slice(i * shard_size, shard_size).sink_parquet(
                SAMPLED_DIR / f"{type}_{i}.parquet",
                compression="zstd",
            )
        )

    print(f"[{type}] {n_rows} rows -> {SHARDS} x ~{shard_size}")

ValueError: negative slice lengths (-999986) are invalid for LazyFrame